# Task 3 — Sentiment vs daily returns

Pipeline:

1. **Normalize timestamps** and roll weekend/holiday news to the **next trading session** (`src/date_alignment.py`).
2. **Score headlines** with VADER (financial short text; no training data required).
3. **Average sentiment** when multiple articles hit the same symbol/day.
4. **Correlate** with **same-day** simple returns on adjusted closes (assignment definition).


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import pearsonr
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.date_alignment import align_news_to_next_trading_day

sns.set_theme(style="whitegrid")

news = pd.read_csv(ROOT / "data" / "raw" / "fnspid_sample.csv")
prices = pd.read_csv(ROOT / "data" / "raw" / "stock_prices_sample.csv")
SYMBOL = "AAPL"

clean_dates = news["date"].astype(str).str.replace(r"\s*UTC-4\s*$", "", regex=True)
news["pub_ts"] = pd.to_datetime(clean_dates, errors="coerce")
news = news.loc[news["stock"] == SYMBOL].dropna(subset=["pub_ts"]).copy()

px = prices.loc[prices["stock"] == SYMBOL, ["Date", "Adj Close"]].copy()
px["Date"] = pd.to_datetime(px["Date"])
px = px.sort_values("Date").drop_duplicates("Date")
trading_days = pd.DatetimeIndex(px["Date"].dt.normalize().unique())

news["trade_date"] = align_news_to_next_trading_day(news["pub_ts"], trading_days)
news = news.dropna(subset=["trade_date"])

analyzer = SentimentIntensityAnalyzer()
news["compound"] = news["headline"].astype(str).map(lambda h: analyzer.polarity_scores(h)["compound"])

daily_sent = news.groupby("trade_date", as_index=False)["compound"].mean().rename(columns={"compound": "sent_mean"})

px = px.set_index("Date")
px["daily_return_pct"] = px["Adj Close"].pct_change() * 100.0

px_out = px[["daily_return_pct"]].reset_index()
px_out["trade_date"] = pd.to_datetime(px_out["Date"]).dt.normalize()
panel = daily_sent.merge(px_out[["trade_date", "daily_return_pct"]], on="trade_date", how="inner")
panel = panel.dropna(subset=["daily_return_pct"])
panel.head()


## Pearson correlation & visuals


In [ ]:
r, p = pearsonr(panel["sent_mean"], panel["daily_return_pct"])
print(f"Pearson r = {r:.3f} (p = {p:.3g})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.scatterplot(data=panel, x="sent_mean", y="daily_return_pct", ax=axes[0], alpha=0.6)
axes[0].set_title(f"{SYMBOL}: sentiment vs same-day return")
axes[0].text(0.05, 0.95, f"r = {r:.3f}", transform=axes[0].transAxes, va="top")

def bucket(x: float) -> str:
    if x >= 0.05:
        return "positive"
    if x <= -0.05:
        return "negative"
    return "neutral"

panel["sent_bucket"] = panel["sent_mean"].map(bucket)
order = ["negative", "neutral", "positive"]
summary = panel.groupby("sent_bucket")["daily_return_pct"].mean().reindex(order)

sns.barplot(x=summary.index.astype(str), y=summary.values, ax=axes[1], palette="RdYlGn", order=order)
axes[1].set_title("Average same-day return by sentiment bucket")
axes[1].set_xlabel("VADER compound bucket")
axes[1].set_ylabel("Mean return (%)")

plt.tight_layout()
plt.show()


## Interpretation (draft for the Medium-style report)

**Strength & direction.** The Pearson coefficient summarizes linear co-movement between the average VADER score and the *same-trading-day* percent change in adjusted close. Values near zero are common in real markets: headline sentiment overlaps with **latent fundamentals**, **macro shocks**, and **microstructure noise**, so a small correlation does not disprove narrative risk — it signals that linear same-day alignment is weak.

**Why VADER?** It is lexicon-based, fast, and transparent — ideal for short headlines. Financial text often contains negation and jargon; for production, fine-tuned finance models (e.g., FinBERT) typically improve precision.

**Limitations.** This notebook uses **contemporaneous** returns; predictively you would lag sentiment or align to **next-day** open-to-close to respect information arrival. Confounders include **simultaneous macro news**, **corporate actions**, and **thin coverage** days. Correlation is **not causation** — use causal framing only with stronger identification.

**Strategy sketch (if future work shows stable edge):** combine **z-scored daily sentiment** with **trend filters** from Task 2 (e.g., trade only when price > 50-day SMA) and enforce **risk caps**; backtest with transaction costs before capital deployment.
